In [1]:
!uv pip install -q "sagemaker>=2.0.0,<3.0.0" "sagemaker[local]" boto3 pytz

In [2]:
!uv pip freeze | grep -E "sagemaker|boto3|pytz"

boto3==1.42.73
pytz==2026.1.post1
sagemaker==2.257.1
sagemaker-core==1.0.77


In [3]:
import os
import subprocess
import sys

In [4]:
!docker build --no-cache -q -t credit-risk-training:latest -f /app/give_me_some_credit/sagemaker/training/Dockerfile /app/give_me_some_credit/sagemaker/training/

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/

sha256:f83b5b36944e74c09c7eed11993d684581867d717a9a6cd220c028bcabfdf88d


In [5]:
!aws s3 ls s3://data-lake --recursive

2026-03-21 20:57:55          0 bronze/
2026-03-21 21:00:00    7564965 bronze/credit_risk/kaggle/2026-03-21/cs-training.csv
2026-03-21 20:58:01          0 feast/registry/
2026-03-21 20:57:59          0 gold/
2026-03-21 21:00:40          0 gold/credit_risk/features/ingestion_date=2026-03-21/
2026-03-21 21:00:40    1662771 gold/credit_risk/features/ingestion_date=2026-03-21/part-00000-3750304a-f45e-43a6-a05b-a2a3080d5fd8.c000.snappy.parquet
2026-03-21 21:00:40    1644117 gold/credit_risk/features/ingestion_date=2026-03-21/part-00001-3750304a-f45e-43a6-a05b-a2a3080d5fd8.c000.snappy.parquet
2026-03-21 20:57:57          0 silver/
2026-03-21 21:00:23          0 silver/credit_risk/cleaned/ingestion_date=2026-03-21/
2026-03-21 21:00:23     998712 silver/credit_risk/cleaned/ingestion_date=2026-03-21/part-00000-9df1b6e3-1810-4262-b4cb-e1b63f3b7dfd.c000.snappy.parquet
2026-03-21 21:00:23     986500 silver/credit_risk/cleaned/ingestion_date=2026-03-21/part-00001-9df1b6e3-1810-4262-b4cb-e1b63f3b7dfd

In [6]:
NETWORK = os.environ["NETWORK"]
result = subprocess.run(
    [
        sys.executable,
        "/app/give_me_some_credit/sagemaker/training/sm_pipeline.py",
        "--mode",
        "local",
        "--ingestion-date",
        "2026-03-21",
        "--s3-endpoint",
        "http://localstack:4566",
        "--n-trials",
        "3",
        "--auc-threshold",
        "0.85",
        "--network",
        NETWORK,
    ],
    capture_output=False,
    text=True,
)
print(f"\nExit code: {result.returncode}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:sagemaker_session:Local SageMaker session ready (bucket=data-lake)
INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for scheduler via: environment_global.
INFO:sm_pipeline:Starting CreditRiskTrainingPipeline...
INFO:sm_pipeline:  mode           : local
INFO:sm_pipeline:  ingestion_date : 2026-03-21
INFO:sm_pipeline:  n_trials       : 3
INFO:sm_pipeline:  auc_threshold  : 0.85
INFO:sm_pipeline:  mlflow_uri     : http://mlflow:5000
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional feat

time="2026-03-21T21:05:59Z" level=warning msg="/tmp/tmpp0znp_81/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-03-21T21:05:59Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmpp0znp_81\".\nSet `external: true` to use an existing network"
 Container 3zcmkdiukk-algo-1-71php Creating 
 Container 3zcmkdiukk-algo-1-71php Created 
Attaching to 3zcmkdiukk-algo-1-71php
 Container 3zcmkdiukk-algo-1-71php Starting 
 Container 3zcmkdiukk-algo-1-71php Started 
3zcmkdiukk-algo-1-71php  | 2026/03/21 21:06:02 INFO mlflow.tracking.fluent: Experiment with name 'credit_risk_pipeline' does not exist. Creating a new experiment.
3zcmkdiukk-algo-1-71php  | 2026-03-21 21:06:02,529 - preprocess - INFO - Reading gold from: /opt/ml/processing/input/gold
3zcmkdiukk-algo-1-71php  | 2026-03-21 21:06:02,562 - preprocess - INFO - Loaded 149390 rows, 18 columns
3zcmkdiukk-algo-1-

INFO:sagemaker.local.image:===== Job Complete =====
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.
INFO:sagemaker.local.entities:Pipeline step 'BaselineTraining' SUCCEEDED.
INFO:sagemaker.local.entities:Starting pipeline step: 'HyperparameterTuning'
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.
INFO:sagemaker.local.local_session:Starting training job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:


fdlhm9vn6l-algo-1-625wz  | 2026-03-21 21:06:29,041 - train_step - INFO - Training baseline: catboost
fdlhm9vn6l-algo-1-625wz  | 2026-03-21 21:06:42,072 - train_step - INFO - [CV-5] AUC=0.8647 ± 0.0015
fdlhm9vn6l-algo-1-625wz  | 2026-03-21 21:06:44,744 - train_step - INFO - [train] AUC=0.8802 KS=0.6028
fdlhm9vn6l-algo-1-625wz  | 2026-03-21 21:06:44,765 - train_step - INFO - [val] AUC=0.8651 KS=0.5848
fdlhm9vn6l-algo-1-625wz  | 2026/03/21 21:06:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
fdlhm9vn6l-algo-1-625wz  | 🏃 View run baseline_catboost at: http://mlflow:5000/#/experiments/1/runs/371ea9ac19254a33bee86fc123fa44d5
fdlhm9vn6l-algo-1-625wz  | 🧪 View experiment at: http://mlflow:5000/#/experiments/1
fdlhm9vn6l-algo-1-625wz  | 2026-03-21 21:06:46,490 - botocore.credentials - INFO - Found credentials in environment variables.
fdlhm9vn6l-algo-1-625wz  | 202

INFO:sagemaker.local.image:===== Job Complete =====
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.
INFO:sagemaker.local.entities:Pipeline step 'HyperparameterTuning' SUCCEEDED.
INFO:sagemaker.local.entities:Starting pipeline step: 'Evaluation'
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.
INFO:sagemaker.local.local_session:Starting processing job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:
    

4sww7xuobb-algo-1-ea9wz  | 2026-03-21 21:08:08,767 - tune_step - INFO - Uploaded tuning_summary.json → s3://data-lake/sagemaker/pipeline/tuning/tuning_summary.json
4sww7xuobb-algo-1-ea9wz  | 2026-03-21 21:08:08,767 - tune_step - INFO - Champion: catboost val_auc=0.8654
4sww7xuobb-algo-1-ea9wz  | 2026-03-21 21:08:08,767 - tune_step - INFO - Tuning complete.
4sww7xuobb-algo-1-ea9wz exited with code 0
 Compose Stopping Aborting on container exit...
 Container 4sww7xuobb-algo-1-ea9wz Stopping 
 Container 4sww7xuobb-algo-1-ea9wz Stopped 
time="2026-03-21T21:08:10Z" level=warning msg="/tmp/tmpbnk0zxk2/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-03-21T21:08:10Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmpbnk0zxk2\".\nSet `external: true` to use an existing network"
 Container durdfst1dg-algo-1-oz89r Creating 
 Container durdfst1dg-algo-1-oz89r Cre